# Decoding & Sampling — Hands-On

**LLM Engineering · Domain 2 · Roadmap Weeks 10/11**

Companion to `02 Literature Notes/LLM Engineering/Decoding and Sampling`. Pure numpy, runs offline.
We treat a fixed logit vector as 'the model' and study how decoding changes what comes out.

## 0. Setup

In [ ]:
%pip install -q numpy
import numpy as np
rng = np.random.RandomState(0)
vocab = ["the","cat","sat","quantum","banana","!"]
logits = np.array([3.0, 2.2, 1.5, 0.3, -0.5, -2.0])
def softmax_T(z, T=1.0):
    z = z/T; e = np.exp(z - z.max()); return e/e.sum()
print("base probs:", np.round(softmax_T(logits),3))

## 1. Temperature reshapes the distribution

In [ ]:
for T in (0.25, 0.5, 1.0, 2.0):
    p = softmax_T(logits, T)
    ent = -(p*np.log(p)).sum()
    print(f"T={T:>4}  {np.round(p,3)}  entropy={ent:.2f}")
print("low T -> peaked/low-entropy (deterministic); high T -> flat/high-entropy")

## 2. The full sampler: temperature + top-k + top-p

In [ ]:
def sample_next(logits, temperature=1.0, top_k=0, top_p=0.0, rng=rng):
    logits = logits.astype(float)
    if temperature <= 0: return int(logits.argmax())
    probs = softmax_T(logits, temperature)
    if top_k and top_k < len(probs):
        keep = np.argsort(probs)[-top_k:]
        m = np.zeros_like(probs, bool); m[keep]=True; probs=np.where(m,probs,0)
    if top_p:
        order = np.argsort(probs)[::-1]; c = np.cumsum(probs[order])
        cut = order[c <= top_p]; cut = cut if len(cut) else order[:1]
        m = np.zeros_like(probs, bool); m[cut]=True; probs=np.where(m,probs,0)
    probs /= probs.sum()
    return int(rng.choice(len(probs), p=probs))

print("greedy     ->", vocab[sample_next(logits, temperature=0)])
print("temp=1.5   ->", vocab[sample_next(logits, temperature=1.5)])
print("top_p=0.9  ->", vocab[sample_next(logits, temperature=1.0, top_p=0.9)])

## 3. Empirical: how often does each strategy pick the rare 'quantum'/'banana'?

In [ ]:
def freq(**kw):
    N=4000; counts=np.zeros(len(vocab))
    for _ in range(N): counts[sample_next(logits, **kw)] += 1
    return counts/N
import numpy as np
for label, kw in {"greedy":dict(temperature=0.0001),
                  "temp=1.0":dict(temperature=1.0),
                  "temp=1.0,top_p=0.9":dict(temperature=1.0, top_p=0.9),
                  "temp=1.0,top_k=2":dict(temperature=1.0, top_k=2)}.items():
    f = freq(**kw)
    rare = f[3]+f[4]     # quantum + banana
    print(f"{label:<22} rare-token rate={rare:.3f}  top pick={vocab[int(f.argmax())]}")

> Nucleus/top-k truncation crush the rare-token rate vs pure temperature sampling — that's the point.

## 4. Structured output wants temperature=0

In [ ]:
# Imagine token 5 ('!') breaks your JSON schema. At temp=0 it is never chosen.
print("greedy picks:", vocab[sample_next(logits, temperature=0)])
bad = sum(sample_next(logits, temperature=1.5)==5 for _ in range(1000))
print(f"at temp=1.5, schema-breaking token chosen {bad}/1000 times")

## 5. Exercises
1. Add a repetition penalty: subtract a constant from logits of already-emitted tokens.
2. Compare top_p=0.5 vs 0.95 rare-token rates.
3. Implement min-p sampling (keep tokens with prob >= min_p * max_prob).
4. Build a tiny autoregressive loop that appends the sampled token and re-scores.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Decoding and Sampling`
- Snippets: `04 Code Snippets/LLM/Temperature Top-k Top-p Sampler`, `.../Temperature Reshapes the Distribution`
- MOC: `06 Maps of Content/LLM Engineering Concepts`